# pyLSTemp - Exemplo de `split_window(...)`

Este notebook demonstra como calcular LST usando metodos split-window.

O fluxo recomendado e:

1. Calcular `brightness(...)` para as bandas termais 10 e 11.
2. Chamar `split_window(...)` usando Red e NIR para NDVI/emissividade interna.
3. Escolher explicitamente `lst_method`, por exemplo `"du-2015"`.

## 1. Instalacao

In [ ]:
# Em Colab/Jupyter, descomente se precisar instalar.
# !pip install --quiet --upgrade git+https://github.com/daciocambraia/pyLSTemp.git rasterio

## 2. Importacoes

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

from pylstemp import brightness, split_window, water_vapor, spectral_index, list_algorithms

import pylstemp
print("pyLSTemp version:", pylstemp.__version__)

## 3. Conferir metodos split-window

In [ ]:
catalog = list_algorithms()
print(list(catalog["split_window"].keys()))

## 4. Ler bandas de entrada

Este exemplo usa:

- `B10`: banda termal 10.
- `B11`: banda termal 11.
- `B4`: Red.
- `B5`: NIR.

In [ ]:
data_dir = Path("data")

band_10_path = data_dir / "LC82210712016107LGN01_B10.tif"
band_11_path = data_dir / "LC82210712016107LGN01_B11.tif"
red_path = data_dir / "LC82210712016107LGN01_B4.tif"
nir_path = data_dir / "LC82210712016107LGN01_B5.tif"

with rasterio.open(band_10_path) as src:
    band_10 = src.read(1).astype(float)
    profile = src.profile.copy()

with rasterio.open(band_11_path) as src:
    band_11 = src.read(1).astype(float)

with rasterio.open(red_path) as src:
    red = src.read(1).astype(float)

with rasterio.open(nir_path) as src:
    nir = src.read(1).astype(float)

print("Shape:", band_10.shape)

## 5. Calcular brightness das bandas 10 e 11

In [ ]:
brightness_10 = brightness(band_10, band="band_10", sensor="landsat_8", mask=(band_10 == 0))
brightness_11 = brightness(band_11, band="band_11", sensor="landsat_8", mask=(band_11 == 0))

print("Brightness 10 mean:", np.nanmean(brightness_10))
print("Brightness 11 mean:", np.nanmean(brightness_11))

## 6. Calcular LST com Du 2015

`du-2015` pode ser usado sem informar `water_vapor`; nesse caso, a biblioteca usa o conjunto de coeficientes geral do artigo.

Para split-window, use emissividade especifica por banda, como `gopinadh-2018` ou `xiaolei-2014`.

In [ ]:
lst_du_celsius = split_window(
    brightness_band_10=brightness_10,
    brightness_band_11=brightness_11,
    band_4_red=red,
    band_5_nir=nir,
    lst_method="du-2015",
    emissivity_method="gopinadh-2018",
    unit="celsius",
)

print("LST Du 2015 min:", np.nanmin(lst_du_celsius))
print("LST Du 2015 max:", np.nanmax(lst_du_celsius))
print("LST Du 2015 mean:", np.nanmean(lst_du_celsius))

## 7. Exemplo opcional com Jimenez-Munoz 2014

`jimenez-munoz-2014` exige `water_vapor`. A celula abaixo estima vapor d'agua com `water_vapor(...)` e passa o raster resultante para `split_window(...)`.

Esta etapa pode ser mais lenta porque o metodo Wang 2015 usa janelas locais.

In [ ]:
# ndvi_image = spectral_index(index="ndvi", nir=nir, red=red, mask=((red == 0) | (nir == 0)))
# water_vapor_image = water_vapor(
#     brightness_band_10=brightness_10,
#     brightness_band_11=brightness_11,
#     ndvi_image=ndvi_image,
#     method="wang-2015",
#     window_size=5,
#     group_count=5,
# )
#
# lst_jimenez_celsius = split_window(
#     brightness_band_10=brightness_10,
#     brightness_band_11=brightness_11,
#     band_4_red=red,
#     band_5_nir=nir,
#     lst_method="jimenez-munoz-2014",
#     emissivity_method="gopinadh-2018",
#     water_vapor=water_vapor_image,
#     unit="celsius",
# )
#
# print("LST Jimenez-Munoz 2014 mean:", np.nanmean(lst_jimenez_celsius))

## 8. Visualizar LST split-window

In [ ]:
valid = lst_du_celsius[np.isfinite(lst_du_celsius)]
vmin, vmax = np.nanpercentile(valid, [2, 98])

plt.figure(figsize=(8, 6))
plt.imshow(lst_du_celsius, cmap="inferno", vmin=vmin, vmax=vmax)
plt.colorbar(label="LST (Celsius)")
plt.title("Split-window LST - Du 2015")
plt.axis("off")
plt.show()

## 9. Salvar resultado opcionalmente

In [ ]:
# output_path = Path("lst_split_window_du_2015_celsius.tif")
# output_profile = profile.copy()
# output_profile.update(dtype="float32", nodata=np.nan, count=1)
#
# with rasterio.open(output_path, "w", **output_profile) as dst:
#     dst.write(lst_du_celsius.astype("float32"), 1)
#
# print("Arquivo salvo em:", output_path)